# Site dependent proportionality and temporal transfer

A portable notebook for the completed filter-only analyses. It reads the included frozen tables and reproduces the specified fits; it does not search external data or fit another model.

**What baseline means.** The frozen study baseline is the saved filter population, flags and analysis rules. The prediction baseline is the training-median HIPS value; proportional prediction is a separate comparison. Neither verifies upstream FTIR spectral baseline correction. See the [coverage map](../coverage_map.md) for the completed reports and pending evidence.

In [ ]:
import sys
import os
from pathlib import Path
hint = os.environ.get('FILTER_ONLY_RELEASE_ROOT')
search = [Path(hint).expanduser().resolve()] if hint else [Path.cwd(), *Path.cwd().parents]
RELEASE_ROOT = next((p for p in search if (p / 'release_config.json').exists()), None)
if RELEASE_ROOT is None:
    raise RuntimeError('Set FILTER_ONLY_RELEASE_ROOT to the extracted release directory.')
sys.path.insert(0, str(RELEASE_ROOT / 'scripts'))
# For notebooks under notebooks/ (e.g. plotting_gaps_scenarios.ipynb):
# sys.path.insert(0, '../research/ftir_hips_chem/scripts')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Config + data
from config import (
    SITES, PROCESSED_SITES_DIR, FILTER_DATA_PATH,
    AERONET_DATA_DIR, WEATHER_DATA_DIR, MAC_VALUE,
)

# Exclusions (use these — do not hand-filter)
from outliers import (
    EXCLUDED_SAMPLES, MANUAL_OUTLIERS,
    apply_exclusion_flags, apply_threshold_flags,
    get_clean_data, print_exclusion_summary,
)

# Data loading / matching
from data_matching import (
    load_aethalometer_data, load_filter_data,
    match_aeth_filter_data, match_all_parameters,
)
from etad_factors import load_etad_factor_contributions, match_etad_factors

# External datasets — these resolve their own location, so do NOT hardcode a
# Drive path. Each checks an env var, then a config constant, then discovers the
# Drive mount: AETHMODULAR_AERONET_DIR / AETHMODULAR_IMPROVE_DIR.
from aeronet import load_aeronet, aeronet_dir, COLS as AERONET_COLS
from improve_io import load_improve_clean

# Plotting — importing the package auto-applies the white-background default
# style (apply_default_style()). Do NOT call plt.style.use('seaborn-v0_8-darkgrid')
# afterwards — it re-adds the grey axes facecolor we don't want.
from plotting import PlotConfig, crossplots, timeseries, distributions, comparisons
from plotting.utils import calculate_regression_stats

PlotConfig.set(sites='all', layout='individual', show_stats=False, show_1to1=False)


The exclusion flags and physical-filter memberships were frozen in the completed analysis. This release verifies their hashes rather than applying new exclusions.

In [ ]:
import importlib.util
from IPython.display import display
spec = importlib.util.spec_from_file_location('release_reproduction', RELEASE_ROOT / 'reproduce.py')
reproduction = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reproduction)
tables = reproduction.run()
from plotting import filter_proportionality as charts
b = tables['proportionality_blocks']
ib = tables['id11_training_blocks']

## Reported products on the same physical filters

In [ ]:
from plotting.filter_diagnostics import relationship
points = pd.read_parquet(RELEASE_ROOT / 'data/diagnostic/analysis_points.parquet')
fig = relationship(points)
display(fig)
plt.close(fig)

**Notes.** The diagnostic cohort includes 545 filters; ratios use 480. Below-MDL predictions remain flagged observations of a reported product, not substituted values.

## Proportionality is site dependent

In [ ]:
fig = charts.paired_blocks(b)
display(fig)
plt.close(fig)

**Notes.** Positive differences favor the intercept. Addis wins all eight withheld quarters; the other sites do not support a uniform preference.

## Limits of later-period prediction

In [ ]:
fig = charts.forward_errors(b)
display(fig)
plt.close(fig)

**Notes.** These folds use only earlier training data and revisit the same record as withheld-quarter evaluation. They are not independent replications.

## ID-11 common-support sensitivity

In [ ]:
fig = charts.id11_errors(ib)
display(fig)
plt.close(fig)

**Notes.** Pooling IDs 11 and 17 is not required for the aggregate intercept advantage: within ID 11, proportional MAE is 9.968 versus 3.884 Mm⁻¹. Restricting training gives a separate, smaller OLS gain on 127 common filters.

## Weighting and directional errors

In [ ]:
fig = charts.weighting(tables['proportionality_summary'])
display(fig)
plt.close(fig)

**Notes.** The Delhi final quarter supplies 68.4% of later-period test filters and 76.6% of intercept-model absolute error. Those are different quantities.

Read the [scientific draft](../manuscript.md), [claim ledger](../claim_ledger.csv), [extended questions](../phase_reports/proportionality_upstream_questions_v2_draft.md) and [candidate evidence](../phase_reports/proportionality_USPA-0257_evidence_package.md).